In [ ]:
import random
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import itertools
import time

In [ ]:
# Problem constants and parameters
num_nodes = 30
num_edges = 15
CONVERGENCE_THRESHOLD = 0.01
TIME_LIMIT_SECONDS = 600



# COLOR_PENALTY = 100
COLOR_PENALTY = 100
HEIGHT_PENALTY = 3

mutation_rate = 0.5
tournament_size = 20
population_size = 100
max_generations = 1500

In [ ]:
# Create a random graph
graph = nx.gnm_random_graph(num_nodes, num_edges)

# Define high nodes and the colors allowed for their height
high_nodes = np.random.choice(num_nodes, size=np.random.randint(1, num_nodes), replace=False)
allowed_height_colors = np.random.choice(range(1, num_nodes + 1), size=len(high_nodes), replace=True)
num_storage_heights = len(high_nodes)  # Number of storage heights equals the number of high nodes; incorrect, revise later

In [ ]:
class Gene:
    def __init__(self, color, material_type, storage_height):
        self.color = color
        self.material_type = material_type
        self.storage_height = storage_height

In [ ]:
class ColoringSolution:
    def __init__(self, chromosome):
        self.chromosome = chromosome
        self.fitness = None


    def calculate_fitness(self, graph, high_nodes, allowed_height_colors, color_weight=1.0, diversity_weight=1.0):
            used_colors = set()
            fitness_value = 0

            for edge in graph.edges:
                node1, node2 = edge
                gene_node1 = self.chromosome[node1]
                gene_node2 = self.chromosome[node2]
                used_colors.add(gene_node1.color)
                used_colors.add(gene_node2.color)

                # Penalize adjacent vertices with the same color
                if gene_node1.color == gene_node2.color:
                    fitness_value += COLOR_PENALTY

                # Penalize a material that does not satisfy the height constraint
                # if gene_node1.storage_height not in allowed_height_colors:
                if not np.any(np.isin(gene_node1.storage_height, allowed_height_colors)):
                    fitness_value += HEIGHT_PENALTY

                # if gene_node2.storage_height not in allowed_height_colors:
                if not np.any(np.isin(gene_node2.storage_height, allowed_height_colors)):
                    fitness_value += HEIGHT_PENALTY

                # Penalize using more colors
                fitness_value += 0.4 * len(used_colors)

            # Objective 1: Minimize the number of colors
            color_objective = len(used_colors)
            # Objective 2: Maximize the diversity of materials stored in high locations
            # diversity_objective = len(set(gene.material_type for gene in self.chromosome if gene.storage_height in allowed_height_colors))
            # diversity_objective = len(set(gene.material_type for gene in self.chromosome if gene.storage_height in allowed_height_colors))
            diversity_objective = len(set(gene.material_type for gene in self.chromosome if np.any(np.isin(gene.storage_height, allowed_height_colors))))


            color_fitness = color_objective * color_weight
            diversity_fitness = diversity_objective * diversity_weight

            fitness_value = color_fitness + diversity_fitness

            return color_objective, diversity_objective, fitness_value

In [ ]:
def initialize_material_type():
    # Define the available material types here
    material_types = ["Type1", "Type2", "Type3"]
    return random.choice(material_types)

In [ ]:
def initialize_population(population_size, num_nodes, num_storage_heights):
    population = []
    for _ in range(population_size):
        chromosome = [
            Gene(
                np.random.randint(1, num_nodes + 1),
                np.random.randint(1, num_storage_heights + 1),
                initialize_material_type()
            ) for _ in range(num_nodes)
        ]
        solution = ColoringSolution(chromosome)
        population.append(solution)
    return population

In [ ]:
def selection(population, tournament_size):
    selected_parents = []

    for _ in range(len(population)):
        tournament = random.sample(population, tournament_size)
        selected_parent = min(tournament, key=lambda solution: solution.fitness[1])
        selected_parents.append(selected_parent)

    return selected_parents

In [ ]:
def one_point_crossover(parent1, parent2):
    crossover_point = np.random.randint(1, len(parent1.chromosome) - 1)
    child1 = np.concatenate((parent1.chromosome[:crossover_point], parent2.chromosome[crossover_point:]))
    child2 = np.concatenate((parent2.chromosome[:crossover_point], parent1.chromosome[crossover_point:]))
    return ColoringSolution(child1), ColoringSolution(child2)

In [ ]:
def bit_mutation(solution, mutation_rate):
    mutated_child = solution.chromosome.copy()
    for i in range(len(mutated_child)):
        if np.random.rand() < mutation_rate:
            # Ensure the new color differs from the colors of adjacent vertices
            adjacent_colors = set(mutated_child[v] for v in graph.neighbors(i))
            available_colors = set(range(1, len(mutated_child) + 1)) - adjacent_colors
            if available_colors:
                mutated_child[i] = random.choice(list(available_colors))
    return ColoringSolution(mutated_child)

In [ ]:
def replace_population(population, new_generation):
    for i in range(len(population)):
        # Check that fitness has been calculated before comparing
        if new_generation[i].fitness is not None and population[i].fitness is not None:
            if new_generation[i].fitness[1] < population[i].fitness[1]:
                population[i] = new_generation[i]

In [ ]:
def check_stop_condition(generation, max_generations, fitness_history, start_time):
    if generation >= max_generations:
        return True

    convergence_generation_count = 10000
    if generation > convergence_generation_count:
        recent_improvements = fitness_history[-convergence_generation_count:]
        if max(recent_improvements) - min(recent_improvements) < CONVERGENCE_THRESHOLD:
            return True

    time_limit_seconds = TIME_LIMIT_SECONDS
    current_time = time.time()
    if current_time - start_time >= time_limit_seconds:
        return True

In [ ]:
def genetic_algorithm(population, graph, high_nodes, allowed_height_colors):
    best_solution = None
    best_num_colors = float('inf')
    best_fitness = float('inf')
    fitness_history = []

    for generation in range(max_generations):
        start_time = time.time()

        for solution in population:
            solution.fitness = solution.calculate_fitness(graph, high_nodes, allowed_height_colors)

        population.sort(key=lambda solution: solution.fitness[1])

        best_index = 0
        current_best_fitness = population[best_index].fitness[1]
        current_best_num_colors = population[best_index].fitness[0]

        if current_best_fitness < best_fitness:
            best_solution = population[best_index]
            best_num_colors = current_best_num_colors
            best_fitness = current_best_fitness

        selected_parents = selection(population, tournament_size)

        new_generation = []
        for i in range(0, len(selected_parents), 2):
            parent1 = selected_parents[i]
            parent2 = selected_parents[i + 1] if i + 1 < len(selected_parents) else selected_parents[i]
            child1, child2 = one_point_crossover(parent1, parent2)
            child1 = bit_mutation(child1, mutation_rate)
            child2 = bit_mutation(child2, mutation_rate)
            new_generation.extend([child1, child2])

        replace_population(population, new_generation)

        fitness_history.append(current_best_fitness)

        if check_stop_condition(generation, max_generations, fitness_history, start_time):
            break

    return best_solution, best_num_colors, best_fitness


In [ ]:
# Test the genetic algorithm
# population = initialize_population(population_size, high_nodes, allowed_height_colors, num_storage_heights)
population = initialize_population(population_size, num_nodes, num_storage_heights)

# def initialize_population(population_size, num_nodes, num_storage_heights):

best_solution, num_colors_used, best_fitness = genetic_algorithm(population, graph, high_nodes, allowed_height_colors)


print("High nodes:")
for node in high_nodes:
    print(f"Node {node + 1}")

print("\nColors allowed for height:")
for i, color in enumerate(allowed_height_colors):
    print(f"Node {high_nodes[i] + 1}: Color {color}")

print("\nSolution:")
for i, color in enumerate(best_solution.chromosome):
    print(f"Node {i + 1}: Color {color}")
print("Number of colors used:", num_colors_used)

# # Example of plotting the graph with distinct colors
# distinct_colors = [plt.cm.rainbow(i / num_colors_used) for i in range(num_colors_used)]
# color_mapping = {original_color: distinct_colors[i] for i, original_color in enumerate(set(best_solution.chromosome))}
# node_colors = [color_mapping[color] for color in best_solution.chromosome]
# position = nx.spring_layout(graph)
# nx.draw(graph, pos=position, node_color=node_colors, with_labels=True)
# plt.show()

/usr/local/lib/python3.10/dist-packages/numpy/lib/arraysetops.py:612: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  mask |= (ar1 == a)


KeyboardInterrupt: ignored